Creating clusters based on distance and time

In [15]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import networkx as nx

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("../data/processed/firms_VIIRS_NOAA20_NRT_south_america_20260521_092524_dbscan.csv")

# -----------------------------
# SPATIAL + TIME FEATURES (already encoded!)
# -----------------------------

coords = np.radians(df[["lat", "lon"]].values)

time_features = df[[
    "time_sin", "time_cos",
    "date_sin", "date_cos"
]].values

# -----------------------------
# SPATIAL INDEX
# -----------------------------
tree = BallTree(coords, metric="haversine")

radius_km = 1.5
radius = radius_km / 6371.0

# -----------------------------
# DISTANCE FUNCTION (CYCLIC TIME)
# -----------------------------
def time_dist(i, j):
    return np.linalg.norm(time_features[i] - time_features[j])

# optional threshold (you can tune this!)
TIME_THRESHOLD = 0.8

# -----------------------------
# GRAPH BUILD
# -----------------------------
G = nx.Graph()
G.add_nodes_from(range(len(df)))

for i in range(len(df)):
    neighbors = tree.query_radius([coords[i]], r=radius)[0]

    for j in neighbors:
        if i == j:
            continue

        if time_dist(i, j) <= TIME_THRESHOLD:
            G.add_edge(i, j)

# -----------------------------
# CONNECTED COMPONENTS
# -----------------------------
components = list(nx.connected_components(G))

labels = np.full(len(df), -1)

for cid, comp in enumerate(components):
    for idx in comp:
        labels[idx] = cid

df["cluster"] = labels

In [16]:
# -----------------------------
# CLUSTER SUMMARY
# -----------------------------
summary = (
    df[df["cluster"] != -1]  # optional: Noise entfernen
    .groupby("cluster")
    .size()
    .reset_index(name="n_points")
    .sort_values("n_points", ascending=False)
)

print(summary)

     cluster  n_points
211      211        37
260      260        31
80        80        17
101      101        16
140      140        13
..       ...       ...
250      250         1
251      251         1
252      252         1
255      255         1
256      256         1

[273 rows x 2 columns]


Cluster visualisation

In [17]:
import folium
import numpy as np
import pandas as pd
import matplotlib.cm as cm

# -----------------------------
# CLUSTER SIZE
# -----------------------------
cluster_sizes = df.groupby("cluster").size()

df["cluster_size"] = df["cluster"].map(cluster_sizes)

# -----------------------------
# COLOR SCALE (based on size)
# -----------------------------
max_size = cluster_sizes.max()
min_size = cluster_sizes.min()

colormap = cm.get_cmap("Reds")  # gut für Fire data

def get_color(size):
    if size is None or np.isnan(size):
        return "#000000"

    # normalisieren 0–1
    norm = (size - min_size) / (max_size - min_size + 1e-9)

    r, g, b, _ = colormap(norm)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"


# -----------------------------
# MAP INIT
# -----------------------------
m = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=5
)

# -----------------------------
# ADD POINTS
# -----------------------------
for _, row in df.iterrows():

    if row["cluster"] == -1:
        color = "#444444"  # noise = grau
    else:
        color = get_color(row["cluster_size"])

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.8
    ).add_to(m)

from pathlib import Path
import webbrowser

out_file = Path("../output/viirs_clusters.html").resolve()
out_file.parent.mkdir(exist_ok=True)

m.save(out_file)

webbrowser.open(f"file://{out_file}")

C:\Users\Jan Krummenacher\AppData\Local\Temp\ipykernel_9792\1497660247.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap("Reds")  # gut für Fire data


True

Convex hull for each cluster

In [18]:
# Function to calculate convex hull area for a cluster
import numpy as np
from scipy.spatial import ConvexHull

def get_convex_hull(points):
    """
    points: Nx2 array (lon, lat)
    returns: hull coordinates (ordered polygon)
    """

    if len(points) < 3:
        return None

    hull = ConvexHull(points)
    return points[hull.vertices]

In [19]:
# Apply function to each cluster
hulls = {}
sizes = df.groupby("cluster").size()

for c in df["cluster"].unique():

    if c == -1:
        continue

    subset = df[df["cluster"] == c]
    points = subset[["lon", "lat"]].values

    hull = get_convex_hull(points)

    if hull is not None:
        hulls[c] = hull

# We map our convex hulls 

In [20]:
import folium
import matplotlib.cm as cm

# Map center
m = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=5
)

max_size = sizes.max()
colormap = cm.get_cmap("Reds")

def get_color(size):
    norm = size / max_size
    r, g, b, _ = colormap(norm)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

# -----------------------------
# DRAW HULLS
# -----------------------------
for cluster_id, hull in hulls.items():

    size = sizes[cluster_id]
    color = get_color(size)

    folium.Polygon(
        locations=[(lat, lon) for lon, lat in hull],
        color=color,
        weight=2,
        fill=True,
        fill_opacity=0.25,
        popup=f"Cluster {cluster_id} | {size} points"
    ).add_to(m)

# -----------------------------
# optional: points overlay
# -----------------------------
for _, row in df.iterrows():

    if row["cluster"] == -1:
        continue

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=2,
        color="black",
        fill=True,
        fill_opacity=0.4
    ).add_to(m)

from pathlib import Path
import webbrowser

out_file = Path("../output/convex_hull_clusters.html").resolve()
out_file.parent.mkdir(exist_ok=True)

m.save(out_file)

webbrowser.open(f"file://{out_file}")


C:\Users\Jan Krummenacher\AppData\Local\Temp\ipykernel_9792\374086243.py:11: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap("Reds")


True